In [1]:
# ============================================================
# Download real fairness datasets for ICDM synthetic fairness paper
# Uses env variable A100 if available
# ============================================================

import os
import sys
import json
import zipfile
import subprocess
from pathlib import Path
from urllib.request import urlretrieve

# ----------------------------
# 0. Base folder
# ----------------------------

BASE = Path(os.environ.get("A100", Path.cwd())).expanduser().resolve()
DATA_DIR = BASE / "fairness_datasets"
RAW_DIR = DATA_DIR / "raw"
PROCESSED_DIR = DATA_DIR / "processed"

RAW_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

print("BASE:", BASE)
print("DATA_DIR:", DATA_DIR)

# ----------------------------
# 1. Helper functions
# ----------------------------

def download_file(url, out_path):
    out_path = Path(out_path)
    out_path.parent.mkdir(parents=True, exist_ok=True)

    if out_path.exists() and out_path.stat().st_size > 0:
        print(f"[OK exists] {out_path}")
        return out_path

    print(f"[DOWNLOADING] {url}")
    urlretrieve(url, out_path)
    print(f"[SAVED] {out_path}")
    return out_path


def ensure_package(pkg):
    try:
        __import__(pkg)
        print(f"[OK package] {pkg}")
    except ImportError:
        print(f"[INSTALLING] {pkg}")
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])


# ----------------------------
# 2. UCI Adult Census
# ----------------------------

adult_dir = RAW_DIR / "adult"
adult_dir.mkdir(exist_ok=True)

adult_urls = {
    "adult.data": "https://archive.ics.uci.edu/ml/machine-learning-databases/adult/adult.data",
    "adult.test": "https://archive.ics.uci.edu/ml/machine-learning-databases/adult/adult.test",
    "adult.names": "https://archive.ics.uci.edu/ml/machine-learning-databases/adult/adult.names",
}

for fname, url in adult_urls.items():
    download_file(url, adult_dir / fname)


# ----------------------------
# 3. COMPAS / ProPublica
# ----------------------------

compas_dir = RAW_DIR / "compas"
compas_dir.mkdir(exist_ok=True)

download_file(
    "https://raw.githubusercontent.com/propublica/compas-analysis/master/compas-scores-two-years.csv",
    compas_dir / "compas-scores-two-years.csv"
)


# ----------------------------
# 4. German Credit
# ----------------------------

german_dir = RAW_DIR / "german_credit"
german_dir.mkdir(exist_ok=True)

german_urls = {
    "german.data": "https://archive.ics.uci.edu/ml/machine-learning-databases/statlog/german/german.data",
    "german.doc": "https://archive.ics.uci.edu/ml/machine-learning-databases/statlog/german/german.doc",
}

for fname, url in german_urls.items():
    download_file(url, german_dir / fname)


# ----------------------------
# 5. Bank Marketing
# ----------------------------

bank_dir = RAW_DIR / "bank_marketing"
bank_dir.mkdir(exist_ok=True)

bank_zip = download_file(
    "https://archive.ics.uci.edu/ml/machine-learning-databases/00222/bank.zip",
    bank_dir / "bank.zip"
)

try:
    with zipfile.ZipFile(bank_zip, "r") as z:
        z.extractall(bank_dir)
    print(f"[EXTRACTED] {bank_zip} -> {bank_dir}")
except Exception as e:
    print("[WARN] Bank zip extraction failed:", e)


# ----------------------------
# 6. Dutch Census via OpenML
#    Sometimes OpenML names differ, so this block tries several names.
# ----------------------------

ensure_package("sklearn")
ensure_package("pandas")

import pandas as pd
from sklearn.datasets import fetch_openml

dutch_dir = RAW_DIR / "dutch_census"
dutch_dir.mkdir(exist_ok=True)

dutch_candidates = [
    "dutch_census",
    "DutchCensus",
    "dutch-census",
    "census-income",
]

dutch_saved = False

for name in dutch_candidates:
    try:
        print(f"[TRY OpenML] {name}")
        ds = fetch_openml(name=name, version=1, as_frame=True)
        df = ds.frame
        out = dutch_dir / f"{name}.csv"
        df.to_csv(out, index=False)
        print(f"[SAVED] {out} shape={df.shape}")
        dutch_saved = True
        break
    except Exception as e:
        print(f"[SKIP] {name}: {str(e)[:200]}")

if not dutch_saved:
    print(
        "[WARN] Dutch Census was not downloaded automatically. "
        "OpenML naming can vary. You can still run experiments without it."
    )


# ----------------------------
# 7. Folktables / ACS benchmarks
#    Downloads ACS person-level data and saves several benchmark tasks.
#    To avoid huge download, starts with selected states.
# ----------------------------

ensure_package("folktables")

from folktables import (
    ACSDataSource,
    ACSIncome,
    ACSEmployment,
    ACSPublicCoverage,
    ACSMobility,
)

acs_dir = RAW_DIR / "acs_folktables"
acs_dir.mkdir(exist_ok=True)

# Change states if needed. CA is large enough for experiments but not insane.
ACS_STATES = ["CA"]
ACS_YEAR = "2018"
ACS_HORIZON = "1-Year"
ACS_SURVEY = "person"

print("[ACS] downloading data...")
data_source = ACSDataSource(
    survey_year=ACS_YEAR,
    horizon=ACS_HORIZON,
    survey=ACS_SURVEY,
    root_dir=str(acs_dir)
)

acs_data = data_source.get_data(states=ACS_STATES, download=True)
print("[ACS] raw shape:", acs_data.shape)

acs_tasks = {
    "ACSIncome": ACSIncome,
    "ACSEmployment": ACSEmployment,
    "ACSPublicCoverage": ACSPublicCoverage,
    "ACSMobility": ACSMobility,
}

for task_name, task in acs_tasks.items():
    try:
        features, labels, groups = task.df_to_pandas(acs_data)

        df = features.copy()
        df["target"] = labels
        df["group"] = groups

        out = PROCESSED_DIR / f"{task_name}_{ACS_YEAR}_{'_'.join(ACS_STATES)}.csv"
        df.to_csv(out, index=False)

        print(f"[SAVED] {task_name}: {out} shape={df.shape}")

    except Exception as e:
        print(f"[WARN] ACS task failed: {task_name}: {e}")


# ----------------------------
# 8. Save metadata for paper
# ----------------------------

metadata = {
    "Adult": {
        "path": str(adult_dir),
        "target": "income >50K",
        "sensitive_attributes": ["sex", "race", "age"],
        "notes": "Classic binary income prediction fairness benchmark."
    },
    "COMPAS": {
        "path": str(compas_dir / "compas-scores-two-years.csv"),
        "target": "two_year_recid",
        "sensitive_attributes": ["race", "sex", "age_cat"],
        "notes": "Criminal justice recidivism fairness benchmark."
    },
    "GermanCredit": {
        "path": str(german_dir / "german.data"),
        "target": "credit risk",
        "sensitive_attributes": ["sex/marital status proxy", "age", "foreign worker"],
        "notes": "Small credit-risk fairness benchmark."
    },
    "BankMarketing": {
        "path": str(bank_dir),
        "target": "term deposit subscription",
        "sensitive_attributes": ["age", "marital", "education"],
        "notes": "Marketing dataset with categorical and continuous attributes."
    },
    "DutchCensus": {
        "path": str(dutch_dir),
        "target": "dataset-dependent",
        "sensitive_attributes": ["sex", "age", "household or occupation-related attributes"],
        "notes": "Downloaded via OpenML if available."
    },
    "Folktables_ACS": {
        "path": str(PROCESSED_DIR),
        "tasks": list(acs_tasks.keys()),
        "states": ACS_STATES,
        "year": ACS_YEAR,
        "sensitive_attributes": ["SEX", "RAC1P", "AGEP"],
        "notes": "Modern ACS-based fairness benchmark tasks."
    }
}

meta_path = DATA_DIR / "dataset_metadata.json"
with open(meta_path, "w", encoding="utf-8") as f:
    json.dump(metadata, f, indent=2, ensure_ascii=False)

print("\nDONE")
print("All datasets folder:", DATA_DIR)
print("Metadata:", meta_path)

BASE: /home/tahiti/DataGenaration
DATA_DIR: /home/tahiti/DataGenaration/fairness_datasets
[DOWNLOADING] https://archive.ics.uci.edu/ml/machine-learning-databases/adult/adult.data
[SAVED] /home/tahiti/DataGenaration/fairness_datasets/raw/adult/adult.data
[DOWNLOADING] https://archive.ics.uci.edu/ml/machine-learning-databases/adult/adult.test
[SAVED] /home/tahiti/DataGenaration/fairness_datasets/raw/adult/adult.test
[DOWNLOADING] https://archive.ics.uci.edu/ml/machine-learning-databases/adult/adult.names
[SAVED] /home/tahiti/DataGenaration/fairness_datasets/raw/adult/adult.names
[DOWNLOADING] https://raw.githubusercontent.com/propublica/compas-analysis/master/compas-scores-two-years.csv
[SAVED] /home/tahiti/DataGenaration/fairness_datasets/raw/compas/compas-scores-two-years.csv
[DOWNLOADING] https://archive.ics.uci.edu/ml/machine-learning-databases/statlog/german/german.data
[SAVED] /home/tahiti/DataGenaration/fairness_datasets/raw/german_credit/german.data
[DOWNLOADING] https://archive.

/home/tahiti/Malashin_Projects/.venv_a100/lib/python3.12/site-packages/sklearn/datasets/_openml.py:109: UserWarning: A network error occurred while downloading https://api.openml.org/api/v1/json/data/list/data_name/dutch_census/limit/2/data_version/1. Retrying...
  warn(


[SKIP] dutch_census: HTTP Error 504: Gateway Time-out
[TRY OpenML] DutchCensus


/home/tahiti/Malashin_Projects/.venv_a100/lib/python3.12/site-packages/sklearn/datasets/_openml.py:109: UserWarning: A network error occurred while downloading https://api.openml.org/api/v1/json/data/list/data_name/dutchcensus/limit/2/data_version/1. Retrying...
  warn(


[SKIP] DutchCensus: HTTP Error 504: Gateway Time-out
[TRY OpenML] dutch-census


/home/tahiti/Malashin_Projects/.venv_a100/lib/python3.12/site-packages/sklearn/datasets/_openml.py:109: UserWarning: A network error occurred while downloading https://api.openml.org/api/v1/json/data/list/data_name/dutch-census/limit/2/data_version/1. Retrying...
  warn(


[SKIP] dutch-census: HTTP Error 504: Gateway Time-out
[TRY OpenML] census-income


/home/tahiti/Malashin_Projects/.venv_a100/lib/python3.12/site-packages/sklearn/datasets/_openml.py:109: UserWarning: A network error occurred while downloading https://api.openml.org/api/v1/json/data/list/data_name/census-income/limit/2/data_version/1. Retrying...
  warn(

KeyboardInterrupt



In [2]:
# ============================================================
# Build processed fairness datasets WITHOUT Dutch Census
# Base: /home/tahiti/DataGenaration/fairness_datasets
# ============================================================

import os
import sys
import subprocess
from pathlib import Path

import pandas as pd
import numpy as np

BASE = Path(os.environ.get("A100", "/home/tahiti/DataGenaration")).expanduser().resolve()
DATA_DIR = BASE / "fairness_datasets"
RAW_DIR = DATA_DIR / "raw"
PROCESSED_DIR = DATA_DIR / "processed"

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

print("BASE:", BASE)
print("DATA_DIR:", DATA_DIR)
print("PROCESSED_DIR:", PROCESSED_DIR)


def ensure_package(pkg):
    try:
        __import__(pkg)
        print(f"[OK package] {pkg}")
    except ImportError:
        print(f"[INSTALLING] {pkg}")
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])


# ============================================================
# 1. Adult Census
# ============================================================

adult_cols = [
    "age", "workclass", "fnlwgt", "education", "education_num",
    "marital_status", "occupation", "relationship", "race", "sex",
    "capital_gain", "capital_loss", "hours_per_week",
    "native_country", "target"
]

adult_train = pd.read_csv(
    RAW_DIR / "adult" / "adult.data",
    header=None,
    names=adult_cols,
    na_values="?",
    skipinitialspace=True
)

adult_test = pd.read_csv(
    RAW_DIR / "adult" / "adult.test",
    header=None,
    names=adult_cols,
    na_values="?",
    skipinitialspace=True,
    comment="|"
)

adult = pd.concat([adult_train, adult_test], axis=0, ignore_index=True)
adult["target"] = adult["target"].astype(str).str.replace(".", "", regex=False).str.strip()
adult["target"] = adult["target"].map({"<=50K": 0, ">50K": 1})
adult = adult.dropna(subset=["target"])
adult["target"] = adult["target"].astype(int)

adult["age_group"] = pd.qcut(adult["age"], q=4, labels=False, duplicates="drop")

out = PROCESSED_DIR / "adult_processed.csv"
adult.to_csv(out, index=False)
print("[SAVED]", out, adult.shape)


# ============================================================
# 2. COMPAS
# ============================================================

compas_path = RAW_DIR / "compas" / "compas-scores-two-years.csv"
compas = pd.read_csv(compas_path)

keep_cols = [
    "age", "age_cat", "sex", "race",
    "priors_count", "c_charge_degree",
    "juv_fel_count", "juv_misd_count", "juv_other_count",
    "two_year_recid"
]

compas = compas[keep_cols].copy()
compas = compas.dropna()
compas = compas.rename(columns={"two_year_recid": "target"})
compas["age_group"] = pd.qcut(compas["age"], q=4, labels=False, duplicates="drop")

out = PROCESSED_DIR / "compas_processed.csv"
compas.to_csv(out, index=False)
print("[SAVED]", out, compas.shape)


# ============================================================
# 3. German Credit
# ============================================================

german_path = RAW_DIR / "german_credit" / "german.data"

german_cols = [
    "checking_status", "duration", "credit_history", "purpose",
    "credit_amount", "savings", "employment", "installment_rate",
    "personal_status_sex", "other_debtors", "residence_since",
    "property", "age", "other_installment_plans", "housing",
    "existing_credits", "job", "num_dependents", "telephone",
    "foreign_worker", "target"
]

german = pd.read_csv(
    german_path,
    sep=" ",
    header=None,
    names=german_cols
)

# UCI: 1 = good credit, 2 = bad credit
german["target"] = german["target"].map({1: 1, 2: 0})
german["age_group"] = pd.qcut(german["age"], q=4, labels=False, duplicates="drop")

out = PROCESSED_DIR / "german_credit_processed.csv"
german.to_csv(out, index=False)
print("[SAVED]", out, german.shape)


# ============================================================
# 4. Bank Marketing
# ============================================================

bank_csv = RAW_DIR / "bank_marketing" / "bank-full.csv"

if not bank_csv.exists():
    # fallback for smaller file
    bank_csv = RAW_DIR / "bank_marketing" / "bank.csv"

bank = pd.read_csv(bank_csv, sep=";")
bank["target"] = bank["y"].map({"no": 0, "yes": 1})
bank = bank.drop(columns=["y"])
bank["age_group"] = pd.qcut(bank["age"], q=4, labels=False, duplicates="drop")

out = PROCESSED_DIR / "bank_marketing_processed.csv"
bank.to_csv(out, index=False)
print("[SAVED]", out, bank.shape)


# ============================================================
# 5. Folktables / ACS
# ============================================================

ensure_package("folktables")

from folktables import (
    ACSDataSource,
    ACSIncome,
    ACSEmployment,
    ACSPublicCoverage,
    ACSMobility,
)

acs_raw_dir = RAW_DIR / "acs_folktables"
acs_raw_dir.mkdir(parents=True, exist_ok=True)

ACS_YEAR = "2018"
ACS_HORIZON = "1-Year"
ACS_SURVEY = "person"

# CA only: enough for first paper experiments, not too huge
ACS_STATES = ["CA"]

print("[ACS] downloading/loading ACS data...")

data_source = ACSDataSource(
    survey_year=ACS_YEAR,
    horizon=ACS_HORIZON,
    survey=ACS_SURVEY,
    root_dir=str(acs_raw_dir)
)

acs_data = data_source.get_data(states=ACS_STATES, download=True)
print("[ACS] raw shape:", acs_data.shape)

acs_tasks = {
    "ACSIncome": ACSIncome,
    "ACSEmployment": ACSEmployment,
    "ACSPublicCoverage": ACSPublicCoverage,
    "ACSMobility": ACSMobility,
}

for task_name, task in acs_tasks.items():
    try:
        features, labels, groups = task.df_to_pandas(acs_data)

        df = features.copy()
        df["target"] = labels.astype(int)
        df["group"] = groups

        if "AGEP" in df.columns:
            df["age_group"] = pd.qcut(df["AGEP"], q=4, labels=False, duplicates="drop")

        out = PROCESSED_DIR / f"{task_name}_{ACS_YEAR}_{'_'.join(ACS_STATES)}_processed.csv"
        df.to_csv(out, index=False)

        print("[SAVED]", out, df.shape)

    except Exception as e:
        print("[WARN] ACS task failed:", task_name, e)


# ============================================================
# 6. Summary
# ============================================================

print("\nDONE. Processed files:")
for p in sorted(PROCESSED_DIR.glob("*.csv")):
    try:
        df_tmp = pd.read_csv(p, nrows=5)
        print("-", p.name, "cols=", len(df_tmp.columns))
    except Exception:
        print("-", p.name)

BASE: /home/tahiti/DataGenaration
DATA_DIR: /home/tahiti/DataGenaration/fairness_datasets
PROCESSED_DIR: /home/tahiti/DataGenaration/fairness_datasets/processed
[SAVED] /home/tahiti/DataGenaration/fairness_datasets/processed/adult_processed.csv (48842, 16)
[SAVED] /home/tahiti/DataGenaration/fairness_datasets/processed/compas_processed.csv (7214, 11)
[SAVED] /home/tahiti/DataGenaration/fairness_datasets/processed/german_credit_processed.csv (1000, 22)
[SAVED] /home/tahiti/DataGenaration/fairness_datasets/processed/bank_marketing_processed.csv (45211, 18)
[INSTALLING] folktables


[ACS] downloading/loading ACS data...

/home/tahiti/DataGenaration/fairness_datasets/raw/acs_folktables/2018/1-Year/csv_pca.zip may be corrupted. Please try deleting it and rerunning this command.

Exception:  File is not a zip file


FileNotFoundError: [Errno 2] No such file or directory: '/home/tahiti/DataGenaration/fairness_datasets/raw/acs_folktables/2018/1-Year/psam_p06.csv'

In [6]:
# ============================================================
# Download and process alternative fairness datasets
# No ACS, no Census, no OpenML
# Base uses env variable A100, fallback: /home/tahiti/DataGenaration
# ============================================================

import os
import sys
import zipfile
import subprocess
from pathlib import Path
from urllib.request import Request, urlopen
from urllib.error import URLError, HTTPError

import pandas as pd
import numpy as np

BASE = Path(os.environ.get("A100", "/home/tahiti/DataGenaration")).expanduser().resolve()
DATA_DIR = BASE / "fairness_datasets"
RAW_DIR = DATA_DIR / "raw"
PROCESSED_DIR = DATA_DIR / "processed"

RAW_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

print("BASE:", BASE)
print("DATA_DIR:", DATA_DIR)
print("RAW_DIR:", RAW_DIR)
print("PROCESSED_DIR:", PROCESSED_DIR)


# ============================================================
# Helpers
# ============================================================

def download_file(url, out_path, min_size=100):
    out_path = Path(out_path)
    out_path.parent.mkdir(parents=True, exist_ok=True)

    if out_path.exists() and out_path.stat().st_size >= min_size:
        print("[OK exists]", out_path, "MB=", round(out_path.stat().st_size / 1024 / 1024, 3))
        return out_path

    print("[DOWNLOADING]", url)

    req = Request(
        url,
        headers={
            "User-Agent": "Mozilla/5.0"
        }
    )

    try:
        with urlopen(req, timeout=120) as r:
            content = r.read()
    except Exception as e:
        print("[FAILED]", url, e)
        raise

    out_path.write_bytes(content)

    if out_path.stat().st_size < min_size:
        print("[WARN] downloaded file is very small:", out_path, out_path.stat().st_size)
        print(out_path.read_bytes()[:300])
        raise RuntimeError(f"Bad download: {out_path}")

    print("[SAVED]", out_path, "MB=", round(out_path.stat().st_size / 1024 / 1024, 3))
    return out_path


def save(df, name):
    out = PROCESSED_DIR / name
    df.to_csv(out, index=False)
    print("[SAVED]", out, df.shape)
    return out


def qcut_safe(s, q=4):
    return pd.qcut(pd.to_numeric(s, errors="coerce"), q=q, labels=False, duplicates="drop")


# ============================================================
# 1. Communities and Crime
# Sensitive-ish: race-related community composition
# Target: high violent crime rate
# ============================================================

comm_dir = RAW_DIR / "communities_crime"
comm_dir.mkdir(exist_ok=True)

comm_data = download_file(
    "https://archive.ics.uci.edu/ml/machine-learning-databases/communities/communities.data",
    comm_dir / "communities.data",
    min_size=1000
)

comm_names = download_file(
    "https://archive.ics.uci.edu/ml/machine-learning-databases/communities/communities.names",
    comm_dir / "communities.names",
    min_size=1000
)

# Standard UCI Communities and Crime has 128 columns.
# First 5 are identifiers/non-predictive, last is ViolentCrimesPerPop.
comm_cols = [
    "state", "county", "community", "communityname", "fold",
    "population", "householdsize", "racepctblack", "racePctWhite",
    "racePctAsian", "racePctHisp", "agePct12t21", "agePct12t29",
    "agePct16t24", "agePct65up", "numbUrban", "pctUrban",
    "medIncome", "pctWWage", "pctWFarmSelf", "pctWInvInc",
    "pctWSocSec", "pctWPubAsst", "pctWRetire", "medFamInc",
    "perCapInc", "whitePerCap", "blackPerCap", "indianPerCap",
    "AsianPerCap", "OtherPerCap", "HispPerCap", "NumUnderPov",
    "PctPopUnderPov", "PctLess9thGrade", "PctNotHSGrad",
    "PctBSorMore", "PctUnemployed", "PctEmploy", "PctEmplManu",
    "PctEmplProfServ", "PctOccupManu", "PctOccupMgmtProf",
    "MalePctDivorce", "MalePctNevMarr", "FemalePctDiv",
    "TotalPctDiv", "PersPerFam", "PctFam2Par", "PctKids2Par",
    "PctYoungKids2Par", "PctTeen2Par", "PctWorkMomYoungKids",
    "PctWorkMom", "NumIlleg", "PctIlleg", "NumImmig",
    "PctImmigRecent", "PctImmigRec5", "PctImmigRec8",
    "PctImmigRec10", "PctRecentImmig", "PctRecImmig5",
    "PctRecImmig8", "PctRecImmig10", "PctSpeakEnglOnly",
    "PctNotSpeakEnglWell", "PctLargHouseFam", "PctLargHouseOccup",
    "PersPerOccupHous", "PersPerOwnOccHous", "PersPerRentOccHous",
    "PctPersOwnOccup", "PctPersDenseHous", "PctHousLess3BR",
    "MedNumBR", "HousVacant", "PctHousOccup", "PctHousOwnOcc",
    "PctVacantBoarded", "PctVacMore6Mos", "MedYrHousBuilt",
    "PctHousNoPhone", "PctWOFullPlumb", "OwnOccLowQuart",
    "OwnOccMedVal", "OwnOccHiQuart", "RentLowQ", "RentMedian",
    "RentHighQ", "MedRent", "MedRentPctHousInc",
    "MedOwnCostPctInc", "MedOwnCostPctIncNoMtg", "NumInShelters",
    "NumStreet", "PctForeignBorn", "PctBornSameState",
    "PctSameHouse85", "PctSameCity85", "PctSameState85",
    "LemasSwornFT", "LemasSwFTPerPop", "LemasSwFTFieldOps",
    "LemasSwFTFieldPerPop", "LemasTotalReq", "LemasTotReqPerPop",
    "PolicReqPerOffic", "PolicPerPop", "RacialMatchCommPol",
    "PctPolicWhite", "PctPolicBlack", "PctPolicHisp",
    "PctPolicAsian", "PctPolicMinor", "OfficAssgnDrugUnits",
    "NumKindsDrugsSeiz", "PolicAveOTWorked", "LandArea",
    "PopDens", "PctUsePubTrans", "PolicCars", "PolicOperBudg",
    "LemasPctPolicOnPatr", "LemasGangUnitDeploy",
    "LemasPctOfficDrugUn", "PolicBudgPerPop",
    "ViolentCrimesPerPop"
]

comm = pd.read_csv(
    comm_data,
    header=None,
    names=comm_cols,
    na_values="?"
)

# Drop non-predictive identifiers.
comm = comm.drop(columns=["state", "county", "community", "communityname", "fold"], errors="ignore")

# Convert numeric.
for c in comm.columns:
    comm[c] = pd.to_numeric(comm[c], errors="coerce")

# Binary target: high violent crime rate above median.
comm["target"] = (comm["ViolentCrimesPerPop"] > comm["ViolentCrimesPerPop"].median()).astype(int)

# Useful sensitive/group columns.
comm["black_group"] = qcut_safe(comm["racepctblack"], q=4)
comm["white_group"] = qcut_safe(comm["racePctWhite"], q=4)
comm["hisp_group"] = qcut_safe(comm["racePctHisp"], q=4)
comm["asian_group"] = qcut_safe(comm["racePctAsian"], q=4)

save(comm, "communities_crime_processed.csv")


# ============================================================
# 2. Default of Credit Card Clients
# Sensitive: SEX, EDUCATION, MARRIAGE, AGE
# Target: default payment next month
# ============================================================

default_dir = RAW_DIR / "default_credit_card"
default_dir.mkdir(exist_ok=True)

default_xls = download_file(
    "https://archive.ics.uci.edu/ml/machine-learning-databases/00350/default%20of%20credit%20card%20clients.xls",
    default_dir / "default_credit_card_clients.xls",
    min_size=100000
)

# Needs xlrd for .xls.
try:
    import xlrd
except ImportError:
    print("[INSTALLING] xlrd")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "xlrd"])

default = pd.read_excel(default_xls, header=1)
default = default.rename(columns={"default payment next month": "target"})
default = default.drop(columns=["ID"], errors="ignore")

# Normalize column names.
default.columns = [str(c).strip().replace(" ", "_") for c in default.columns]

if "AGE" in default.columns:
    default["age_group"] = qcut_safe(default["AGE"], q=4)

save(default, "default_credit_card_processed.csv")


# ============================================================
# 3. Student Performance
# Sensitive: sex, age, address, famsize, Pstatus
# Target: pass/fail based on final grade G3
# ============================================================

student_dir = RAW_DIR / "student_performance"
student_dir.mkdir(exist_ok=True)

student_zip = download_file(
    "https://archive.ics.uci.edu/ml/machine-learning-databases/00320/student.zip",
    student_dir / "student.zip",
    min_size=1000
)

with zipfile.ZipFile(student_zip, "r") as z:
    z.extractall(student_dir)

student_files = [
    student_dir / "student-mat.csv",
    student_dir / "student-por.csv"
]

dfs = []
for f in student_files:
    if f.exists():
        df = pd.read_csv(f, sep=";")
        df["subject"] = f.stem.replace("student-", "")
        dfs.append(df)

student = pd.concat(dfs, axis=0, ignore_index=True)

# Binary target: pass if final grade G3 >= 10.
student["target"] = (pd.to_numeric(student["G3"], errors="coerce") >= 10).astype(int)
student["age_group"] = qcut_safe(student["age"], q=4)

save(student, "student_performance_processed.csv")


# ============================================================
# 4. Heart Disease Cleveland
# Sensitive: sex, age
# Target: disease presence
# ============================================================

heart_dir = RAW_DIR / "heart_disease"
heart_dir.mkdir(exist_ok=True)

heart_data = download_file(
    "https://archive.ics.uci.edu/ml/machine-learning-databases/heart-disease/processed.cleveland.data",
    heart_dir / "processed.cleveland.data",
    min_size=1000
)

heart_cols = [
    "age", "sex", "cp", "trestbps", "chol", "fbs", "restecg",
    "thalach", "exang", "oldpeak", "slope", "ca", "thal", "num"
]

heart = pd.read_csv(
    heart_data,
    header=None,
    names=heart_cols,
    na_values="?"
)

for c in heart.columns:
    heart[c] = pd.to_numeric(heart[c], errors="coerce")

# Binary target: 0 = no disease, 1 = disease.
heart["target"] = (heart["num"] > 0).astype(int)
heart["age_group"] = qcut_safe(heart["age"], q=4)

save(heart, "heart_disease_processed.csv")


# ============================================================
# 5. Law School GPA / LSAC
# Try several public mirrors.
# Sensitive: race, sex
# Target: pass_bar / high GPA depending on available columns
# ============================================================

law_dir = RAW_DIR / "law_school"
law_dir.mkdir(exist_ok=True)

law_urls = [
    "https://raw.githubusercontent.com/Trusted-AI/AIF360/main/aif360/data/raw/law_school/law_school_clean.csv",
    "https://raw.githubusercontent.com/Trusted-AI/AIF360/master/aif360/data/raw/law_school/law_school_clean.csv",
    "https://raw.githubusercontent.com/algofairness/fairness-comparison/master/fairness/data/preprocessed/lawschool.csv",
]

law_path = None

for i, url in enumerate(law_urls):
    try:
        p = download_file(url, law_dir / f"law_school_{i}.csv", min_size=1000)
        # Quick check: skip HTML 404 pages.
        head = p.read_text(errors="ignore")[:200].lower()
        if "<html" in head or "404:" in head:
            print("[BAD MIRROR]", url)
            p.unlink(missing_ok=True)
            continue
        law_path = p
        break
    except Exception as e:
        print("[SKIP LAW URL]", url, e)

if law_path is not None:
    law = pd.read_csv(law_path)

    # Clean column names.
    law.columns = [str(c).strip().replace(" ", "_") for c in law.columns]

    # Try to define target robustly.
    cols_lower = {c.lower(): c for c in law.columns}

    if "pass_bar" in cols_lower:
        law["target"] = pd.to_numeric(law[cols_lower["pass_bar"]], errors="coerce").fillna(0).astype(int)
    elif "bar" in cols_lower:
        law["target"] = pd.to_numeric(law[cols_lower["bar"]], errors="coerce").fillna(0).astype(int)
    elif "zfygpa" in cols_lower:
        c = cols_lower["zfygpa"]
        law["target"] = (pd.to_numeric(law[c], errors="coerce") > pd.to_numeric(law[c], errors="coerce").median()).astype(int)
    elif "ugpa" in cols_lower:
        c = cols_lower["ugpa"]
        law["target"] = (pd.to_numeric(law[c], errors="coerce") > pd.to_numeric(law[c], errors="coerce").median()).astype(int)
    else:
        print("[WARN] Law School loaded, but target column not recognized. Saving without target.")

    save(law, "law_school_processed.csv")
else:
    print("[WARN] Law School was not downloaded. Continue without it.")


# ============================================================
# 6. Final summary
# ============================================================

print("\nDONE. All processed files:")
for p in sorted(PROCESSED_DIR.glob("*.csv")):
    try:
        df = pd.read_csv(p, nrows=3)
        print("-", p.name, "| cols:", len(df.columns))
    except Exception as e:
        print("-", p.name, "|", e)

BASE: /home/tahiti/DataGenaration
DATA_DIR: /home/tahiti/DataGenaration/fairness_datasets
RAW_DIR: /home/tahiti/DataGenaration/fairness_datasets/raw
PROCESSED_DIR: /home/tahiti/DataGenaration/fairness_datasets/processed
[DOWNLOADING] https://archive.ics.uci.edu/ml/machine-learning-databases/communities/communities.data
[SAVED] /home/tahiti/DataGenaration/fairness_datasets/raw/communities_crime/communities.data MB= 1.052
[DOWNLOADING] https://archive.ics.uci.edu/ml/machine-learning-databases/communities/communities.names
[SAVED] /home/tahiti/DataGenaration/fairness_datasets/raw/communities_crime/communities.names MB= 0.026
[SAVED] /home/tahiti/DataGenaration/fairness_datasets/processed/communities_crime_processed.csv (1994, 128)
[DOWNLOADING] https://archive.ics.uci.edu/ml/machine-learning-databases/00350/default%20of%20credit%20card%20clients.xls


/tmp/ipykernel_757750/2178989760.py:162: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  comm["target"] = (comm["ViolentCrimesPerPop"] > comm["ViolentCrimesPerPop"].median()).astype(int)
/tmp/ipykernel_757750/2178989760.py:165: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  comm["black_group"] = qcut_safe(comm["racepctblack"], q=4)
/tmp/ipykernel_757750/2178989760.py:166: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joinin

[SAVED] /home/tahiti/DataGenaration/fairness_datasets/raw/default_credit_card/default_credit_card_clients.xls MB= 5.283
[INSTALLING] xlrd
[SAVED] /home/tahiti/DataGenaration/fairness_datasets/processed/default_credit_card_processed.csv (30000, 25)
[DOWNLOADING] https://archive.ics.uci.edu/ml/machine-learning-databases/00320/student.zip
[SAVED] /home/tahiti/DataGenaration/fairness_datasets/raw/student_performance/student.zip MB= 0.02
[SAVED] /home/tahiti/DataGenaration/fairness_datasets/processed/student_performance_processed.csv (1044, 36)
[DOWNLOADING] https://archive.ics.uci.edu/ml/machine-learning-databases/heart-disease/processed.cleveland.data
[SAVED] /home/tahiti/DataGenaration/fairness_datasets/raw/heart_disease/processed.cleveland.data MB= 0.018
[SAVED] /home/tahiti/DataGenaration/fairness_datasets/processed/heart_disease_processed.csv (303, 16)
[DOWNLOADING] https://raw.githubusercontent.com/Trusted-AI/AIF360/main/aif360/data/raw/law_school/law_school_clean.csv
[FAILED] https: